In [1]:

# Imports
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import mlflow

c:\Users\USER\anaconda3\envs\ecg_project\lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


In [5]:
# -----------------------------
# Paramètres
# -----------------------------
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Résolution robuste du dossier processed (remonte l'arborescence depuis le cwd)
def find_processed_dir():
    cur = os.getcwd()
    while True:
        candidate = os.path.join(cur, "data", "processed")
        if os.path.isdir(candidate):
            return os.path.abspath(candidate)
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    # fallback: try relative to this notebook file location if possible
    # (Jupyter notebooks may start with different working dirs)
    possible = os.path.abspath(os.path.join('..', '..', 'data', 'processed'))
    if os.path.isdir(possible):
        return possible
    raise FileNotFoundError(f'Could not find data/processed starting from cwd={os.getcwd()}')

PROCESSED_DIR = find_processed_dir()
NUM_CLASSES = 2
IMAGE_SIZE = 224

In [3]:
# -----------------------------
# Chargement des données
# -----------------------------
images = np.load(os.path.join(PROCESSED_DIR, "images.npy"))  # (N,224,224,1)
labels = np.load(os.path.join(PROCESSED_DIR, "labels.npy"))

# Corriger dimensions
images = images.squeeze(-1)  # (N,224,224)

X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)  # (N,1,224,224)
y = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batch:", next(iter(train_loader))[0].shape)

Train batch: torch.Size([16, 1, 224, 224])


In [4]:
# -----------------------------
# Définition du CNN
# -----------------------------
class ECG_CNN(nn.Module):
    def __init__(self, num_classes=2):
        super(ECG_CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 112x112

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 56x56

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28x28

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*14*14, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = ECG_CNN(num_classes=NUM_CLASSES).to(DEVICE)

In [5]:
# -----------------------------
# Loss & Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [6]:
# -----------------------------
# MLflow
# -----------------------------
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("CNN_ECG")

if mlflow.active_run():
    mlflow.end_run()


2026/02/01 11:01:52 INFO mlflow.tracking.fluent: Experiment with name 'CNN_ECG' does not exist. Creating a new experiment.


In [7]:
# -----------------------------
# Entraînement
# -----------------------------
with mlflow.start_run(run_name="CNN_ECG_Run"):

    mlflow.log_param("model", "CNN_ECG")
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)

    for epoch in range(EPOCHS):
        # ===== Train =====
        model.train()
        train_loss, train_correct = 0.0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            train_correct += (outputs.argmax(1) == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_acc = train_correct / len(train_loader.dataset)

        # ===== Validation =====
        model.eval()
        val_loss, val_correct = 0.0, 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_correct += (outputs.argmax(1) == targets).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)

        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)

    # -----------------------------
    # Sauvegarde
    # -----------------------------
    mlflow.pytorch.log_model(model, "cnn_ecg")

print("✅ Entraînement CNN terminé avec succès")

Epoch [1/15] Train Loss: 0.6992 | Train Acc: 0.5036 Val Loss: 0.6933 | Val Acc: 0.4712
Epoch [2/15] Train Loss: 0.6934 | Train Acc: 0.5448 Val Loss: 0.6916 | Val Acc: 0.5288
Epoch [3/15] Train Loss: 0.6905 | Train Acc: 0.5375 Val Loss: 0.6920 | Val Acc: 0.5288
Epoch [4/15] Train Loss: 0.6920 | Train Acc: 0.5545 Val Loss: 0.6933 | Val Acc: 0.5288
Epoch [5/15] Train Loss: 0.6912 | Train Acc: 0.5545 Val Loss: 0.6915 | Val Acc: 0.5288
Epoch [6/15] Train Loss: 0.6876 | Train Acc: 0.5545 Val Loss: 0.6937 | Val Acc: 0.5288
Epoch [7/15] Train Loss: 0.6910 | Train Acc: 0.5545 Val Loss: 0.6922 | Val Acc: 0.5288
Epoch [8/15] Train Loss: 0.6878 | Train Acc: 0.5545 Val Loss: 0.6926 | Val Acc: 0.5288
Epoch [9/15] Train Loss: 0.6920 | Train Acc: 0.5545 Val Loss: 0.6914 | Val Acc: 0.5288
Epoch [10/15] Train Loss: 0.6902 | Train Acc: 0.5569 Val Loss: 0.6915 | Val Acc: 0.5288
Epoch [11/15] Train Loss: 0.6902 | Train Acc: 0.5545 Val Loss: 0.6915 | Val Acc: 0.5288
Epoch [12/15] Train Loss: 0.6904 | Train 